## 🏗️ Core Concept: SQL Common Table Expressions (CTEs)
---

<div style="padding: 15px; border-left: 8px solid #667eea; background-color: #f2f5fd; color: #1e2646; border-radius: 4px;">
    <strong>The Core Insight:</strong> A CTE (<code>WITH</code> clause) creates a temporary, named result set that exists purely for the duration of a single execution. It replaces deeply nested subqueries, vastly improving readability and maintainability.
</div>

### 🛠️ The Mechanics

Instead of reading SQL from the "inside-out" (subqueries), CTEs allow you to read SQL from "top-to-bottom" (sequential data pipelines).

---

### 🏢 Real-World Mental Model: The Assembly Line
Imagine building a car. You don't build the engine, chassis, and interior simultaneously in one big pile (a giant nested subquery). 

Instead, you have stations (CTEs):
1. **Station 1:** Build the chassis.
2. **Station 2:** Build the engine (maybe it needs the chassis specs).
3. **Final Assembly:** Put them all together.

CTEs let you chain logical transformation steps exactly like an assembly line.

In [ ]:
# Setup DuckDB for testing
import duckdb
import pandas as pd

conn = duckdb.connect(':memory:')

conn.execute("""
    CREATE TABLE sales (
        region VARCHAR,
        product VARCHAR,
        amount DECIMAL,
        date DATE
    );
    INSERT INTO sales VALUES 
        ('North', 'Widget', 100, '2023-01-01'),
        ('North', 'Widget', 150, '2023-01-02'),
        ('South', 'Gadget', 200, '2023-01-01'),
        ('South', 'Widget', 50, '2023-01-03'),
        ('East', 'Gadget', 300, '2023-01-01');
""")

## 🐢 Approach 1: The Nested Subquery (The Old Way)

Let's say we want to find all regions where the *average sales* are greater than $120.

To read this, your eyes have to jump to the middle, figure out what it's doing, and then jump back out to see how the outer query uses it.

In [ ]:
subquery_sql = """
SELECT 
    region,
    avg_amount
FROM (
    SELECT 
        region,
        AVG(amount) as avg_amount
    FROM sales
    GROUP BY region
) sub
WHERE avg_amount > 120;
"""

print(conn.execute(subquery_sql).df())

## 🚀 Approach 2: The CTE (The Modern Way)

We declare the logic upfront. It reads exactly like a variable assignment in Python.

In [ ]:
cte_sql = """
WITH RegionAverages AS (
    SELECT 
        region,
        AVG(amount) as avg_amount
    FROM sales
    GROUP BY region
)
SELECT 
    region,
    avg_amount
FROM RegionAverages
WHERE avg_amount > 120;
"""

print(conn.execute(cte_sql).df())

## 🧬 Anatomy of a CTE

```sql
WITH cte_name_1 AS (
    -- Query logic here
),
cte_name_2 AS (
    -- You can reference cte_name_1 here!
)
SELECT * FROM cte_name_2;
```

### Why is that second part magical?
**Chaining.** CTEs can reference other CTEs defined *above* them in the same `WITH` clause. This allows you to break down extremely complex transformations into a sequence of simple, testable steps.

In [ ]:
chained_cte_sql = """
WITH RegionalTotals AS (
    SELECT region, SUM(amount) as total_sales
    FROM sales
    GROUP BY region
),
CompanyTotal AS (
    SELECT SUM(total_sales) as grand_total
    FROM RegionalTotals
)
-- Now we use both!
SELECT 
    rt.region,
    rt.total_sales,
    ROUND((rt.total_sales / ct.grand_total) * 100, 2) as percent_of_total
FROM RegionalTotals rt
CROSS JOIN CompanyTotal ct
ORDER BY percent_of_total DESC;
"""

print(conn.execute(chained_cte_sql).df())

## 🎤 Interview Q&A

### Q1: Does a CTE improve performance compared to a subquery?

**Answer:** Generally, no. Modern query optimizers (like PostgreSQL or Snowflake) will usually compile a CTE and a Subquery into the exact same execution plan. The primary benefit is **developer ergonomics** and readability. However, in some databases (like older Postgres versions), CTEs act as an optimization fence, which forces materialization and can actually *hurt* performance if used incorrectly.

---

### Q2: What is a Recursive CTE?

**Answer:** A recursive CTE (`WITH RECURSIVE`) is a CTE that references itself. It's used exclusively for traversing hierarchical or tree-structured data natively in SQL, like organizational charts (managers of managers) or bill-of-materials.

---

### Q3: Can you UPDATE or DELETE using a CTE?

**Answer:** Yes! In Postgres, CTEs aren't strictly limited to `SELECT` statements. You can chain data manipulation. For example, you can write a CTE that `DELETE`s older rows, uses the `RETURNING` clause to grab those deleted IDs, and passes them to a subsequent `INSERT` statement to move them to an archive table securely in one query.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **CTE (Common Table Expression)** | A temporary, named result set defined via `WITH` that exists only during query execution. |
| **Subquery** | A query nested inside another query (e.g., inside the `FROM` or `WHERE` clause). Often harder to read than CTEs. |
| **Optimization Fence** | A database engine behavior where the engine is forced to execute a CTE fully before proceeding, rather than folding its logic into the main query plan. |

In [ ]:
# -------------------------------------------------------
# PRACTICE: Write a CTE
# 1. Create a CTE named 'WidgetSales' that filters only for product='Widget'.
# 2. Write a main query that selects the region and the MAX amount from WidgetSales.
# -------------------------------------------------------

practice_sql = """
-- Write your CTE here
"""

# Uncomment to test
# print(conn.execute(practice_sql).df())